# Scaling and Distance

**Project question:** Which customer looks nearest when the variables use incompatible units?

By the end of this notebook, you should be able to:

- explain how a large numerical scale can dominate Euclidean distance
- standardize features before comparing distances
- recognize that scaling does not make every feature substantively relevant

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [1]:

from lite_setup import ensure_packages
await ensure_packages()

Using the current Python environment.


In [2]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

In [4]:
df = pd.read_csv(DATA / 'simulated_customer_profiles.csv')
features = ['visits_per_month', 'avg_order_value', 'discount_rate', 'email_opens', 'tenure_months', 'support_contacts', 'returns_per_year']
X = df[features]
X.describe()

,visits_per_month,avg_order_value,discount_rate,email_opens,tenure_months,support_contacts,returns_per_year
count,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000
mean,9.137037,42.682000,0.233667,4.922222,20.559259,1.796296,1.311111
std,4.764307,20.806163,0.127081,3.476617,9.468114,1.120602,0.916326
min,1.000000,5.000000,0.042000,0.000000,1.000000,0.000000,0.000000
25%,5.000000,26.110000,0.130000,2.000000,13.000000,1.000000,1.000000
50%,9.000000,35.190000,0.184000,5.000000,21.000000,2.000000,1.000000
75%,13.000000,62.940000,0.362250,8.000000,29.000000,3.000000,2.000000
max,19.000000,94.460000,0.527000,13.000000,43.000000,5.000000,5.000000


In [5]:
raw_dist = pairwise_distances(X)
scaler = StandardScaler().fit(X)
scaled = scaler.transform(X)
scaled_dist = pairwise_distances(scaled)
np.fill_diagonal(raw_dist, np.inf)
np.fill_diagonal(scaled_dist, np.inf)
query = 0
raw_neighbor = int(np.argmin(raw_dist[query]))
scaled_neighbor = int(np.argmin(scaled_dist[query]))
pd.DataFrame([
    {
        'space': 'raw units',
        'query_customer': df.loc[query, 'customer_id'],
        'nearest_customer': df.loc[raw_neighbor, 'customer_id'],
        'distance': raw_dist[query, raw_neighbor],
    },
    {
        'space': 'standardized',
        'query_customer': df.loc[query, 'customer_id'],
        'nearest_customer': df.loc[scaled_neighbor, 'customer_id'],
        'distance': scaled_dist[query, scaled_neighbor],
    },
])

,space,query_customer,nearest_customer,distance
0,raw units,P0001,P0250,3.163432
1,standardized,P0001,P0163,0.613107


In [6]:
scale_table = pd.DataFrame({'feature': features, 'std_dev': X.std().values}).sort_values('std_dev', ascending=False)
scale_table

,feature,std_dev
1,avg_order_value,20.806163
4,tenure_months,9.468114
0,visits_per_month,4.764307
3,email_opens,3.476617
5,support_contacts,1.120602
6,returns_per_year,0.916326
2,discount_rate,0.127081


**Interpretation:** In raw units, features such as average order value or tenure can dominate features measured on smaller scales. Standardization gives each feature variance one in this sample, so the nearest neighbor can change.

Scaling is usually necessary for KNN, K-means, PCA on a correlation scale, and penalized regression. It does not decide whether Euclidean distance is meaningful, whether features deserve equal weight, or whether outliers need robust treatment.